# 01. Минимальный Deep Agent

Самый маленький слой: модель, системный промпт и `create_deep_agent`.
Здесь ещё нет кастомных tools, backend, skills и subagents.

После выполнения этой ячейки сгенерируется `agents/step_01_minimal.py` —
самостоятельный LangGraph entrypoint. Ты сможешь запустить его через:

```bash
uv run langgraph dev --config langgraph.steps.json
```

и в Deep Agents UI использовать Assistant ID: `openclaw`.

## Визуальная рамка

![Эволюция агента](visuals/01_agent_evolution.svg)

Этот шаг соответствует слайдам 02-03: от чатбота и tool-calling к Deep Agent как связке `LLM + tools + state + loop + human checkpoint`.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'pyproject.toml').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)


BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.

If local shell execution is unavailable, use filesystem tools and clearly state
which verification would require shell access.
"""


agent = create_deep_agent(
    model=model_name(),
    tools=[],
    system_prompt=BASE_PROMPT,
)

print(type(agent).__name__)

### Сгенерировать entrypoint для LangGraph

Эта ячейка создаёт `agents/step_01_minimal.py` — файл, который
`langgraph dev` может загрузить как граф.

In [ ]:
import json

AGENTS_DIR = REPO_ROOT / "agents"
AGENTS_DIR.mkdir(exist_ok=True)

agent_code = '''"""Step 01: minimal agent. Model + system prompt only."""

from __future__ import annotations

import os

from deepagents import create_deep_agent
from dotenv import load_dotenv

load_dotenv()

agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", "openrouter:tencent/hy3:free"),
    tools=[],
    system_prompt="""\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
""",
)
'''

step_file = AGENTS_DIR / "step_01_minimal.py"
step_file.write_text(agent_code)

steps_config = {
    "dependencies": ["."],
    "graphs": {
        "openclaw": "./agents/step_01_minimal.py:agent",
    },
    "env": ".env",
}
config_path = REPO_ROOT / "langgraph.steps.json"
config_path.write_text(json.dumps(steps_config, indent=2) + "\n")

print(f"Wrote {step_file}")
print(f"Wrote {config_path}")
print()
print("Run:")
print("  uv run langgraph dev --config langgraph.steps.json")

### Проверочный prompt

После запуска `langgraph dev` попробуй в Deep Agents UI:

> "Объясни, как ты будешь подходить к задаче в незнакомом репозитории."